In [ ]:
import json

with open('/content/BLOTTER.json','r') as file:
  data = json.load(file)

print(data)

[['US U6', 15.0, '111-22'], ['TY U6', 25.0, '109-04+'], ['SR 15Y', -6.0, 4.272], ['US U6', 15.0, '111-18'], ['US U6', 15.0, '111-15'], ['US U6', 15.0, '111-12'], ['TY U6', 25.0, '109-02+'], ['TY U6', 25.0, '109-00+'], ['US U6', 15.0, '111-09'], ['US U6', -25.0, '111-16'], ['TY U6', -25.0, '109-05+'], ['US U6', 15.0, '111-15'], ['TY U6', 25.0, '109-02+'], ['SR 15Y', 6.0, 4.2895], ['SR 5Y', 12.0, 3.999], ['SR 5Y', 7.0, 4.0], ['TY U6', 25.0, '109-00+'], ['USTB 3605_4.375', -25.0, '98-181'], ['SR 10Y', 25.0, 4.14333], ['', '', ''], ['TY U6', 75.0, '108-28+'], ['TY U6', -50.0, '109-06'], ['SR 15Y', -4.496, 4.2765], ['SR 15Y', -4.493, 4.266], ['SR 10Y', -6.103, 4.117], ['TY U6', -50.0, '109-07'], ['TY U6', -50.0, '109-08+'], ['SR 15Y', -7.0, 4.261], ['US U6', 25.0, '111-25'], ['TY U6', 25.0, '109-05+'], ['US U6', 25.0, '111-23'], ['US U6', 25.0, '111-20'], ['TY U6', 25.0, '109-02+'], ['TY U6', 75.0, '109-01+'], ['TY U6', -50.0, '109-01'], ['SR 15Y', 7.0, 4.2805], ['', '', ''], ['TY U6', -50.

In [ ]:
import pandas as pd

# Create a DataFrame from the data
df = pd.DataFrame(data, columns=['Ticker Name', 'Notional', 'Size'])

# Filter out rows where 'Ticker Name' is empty
df_cleaned = df[df['Ticker Name'] != ''].reset_index(drop=True)

#display(df_cleaned)

df_cleaned.head()

,Ticker Name,Notional,Size
0,US U6,15.0,111-22
1,TY U6,25.0,109-04+
2,SR 15Y,-6.0,4.272
3,US U6,15.0,111-18
4,US U6,15.0,111-15


In [ ]:
# Export the cleaned DataFrame to an Excel file
excel_file_path = '/content/BLOTTER_cleaned.xlsx'
df_cleaned.to_excel(excel_file_path, index=False)

print(f"DataFrame exported to {excel_file_path}")

DataFrame exported to /content/BLOTTER_cleaned.xlsx


In [ ]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 54.6 MB/s eta 0:00:00


In [ ]:
%%writefile blotter_app.py

import streamlit as st
import pandas as pd
import json

# --- Load and Prepare Data ---
@st.cache_data
def load_data():
    with open('/content/BLOTTER.json', 'r') as file:
        data = json.load(file)
    df = pd.DataFrame(data, columns=['Ticker Name', 'Notional', 'Size'])

    # Fix: Correctly filter out rows where 'Ticker Name' is empty
    df_cleaned = df[df['Ticker Name'] != ''].reset_index(drop=True)

    # Fix: Correct syntax for pd.to_numeric errors argument
    df_cleaned['Notional'] = pd.to_numeric(df_cleaned['Notional'], errors='coerce')

    return df_cleaned

df_cleaned = load_data()

# --- Streamlit App Layout ---
st.set_page_config(layout="wide")
st.title('Interactive Blotter Analysis')

st.markdown("""
This interactive dashboard allows you to explore and filter the blotter data.
Use the sidebar controls to narrow down your view.
""")

# --- Sidebar Filters ---
st.sidebar.header('Filter Options')

# Ticker Name Filter
ticker_options = df_cleaned['Ticker Name'].unique().tolist()
selected_tickers = st.sidebar.multiselect(
    'Select Ticker Name(s)',
    options=ticker_options,
    default=ticker_options
)

# Notional Range Filter
min_notional = df_cleaned['Notional'].min()
max_notional = df_cleaned['Notional'].max()
notional_range = st.sidebar.slider(
    'Notional Range',
    min_value=float(min_notional),
    max_value=float(max_notional),
    value=(float(min_notional), float(max_notional))
)

# --- Apply Filters ---
# Fix: Correctly apply filters and define 'filtered_df'
filtered_df = df_cleaned[df_cleaned['Ticker Name'].isin(selected_tickers)]
filtered_df = filtered_df[
    (filtered_df['Notional'] >= notional_range[0]) &
    (filtered_df['Notional'] <= notional_range[1])
]

# --- Display Results ---
st.subheader('Filtered Blotter Data')
st.dataframe(filtered_df, use_container_width=True)

st.subheader('Summary Statistics')
st.write(filtered_df.describe())

Overwriting blotter_app.py


In [27]:
import subprocess
import sys

# Ensure Streamlit is installed
!{sys.executable} -m pip install streamlit --quiet

# Run the Streamlit app in the background
# Streamlit typically runs on port 8501 by default
print('Starting Streamlit app...')
process = subprocess.Popen([
    'streamlit', 'run', 'blotter_app.py',
    '--server.port', '8501',
    '--server.enableCORS', 'true',
    '--server.enableXsrfProtection', 'false'
])

# Note: Colab usually provides a public URL for exposed ports automatically.
# You should see a link pop up in the output of this cell or in a separate tab/window.
# If not, you might need to manually check the 'Ports' tab in the right-hand panel of Colab
# (next to 'Files', 'Table of contents', etc.) once the Streamlit app is running.
print(f"Streamlit app running. Check the output above for a public URL or look for a 'Ports' tab in Colab.")

Starting Streamlit app...
Streamlit app running. Check the output above for a public URL or look for a 'Ports' tab in Colab.


After running the cell above, you should see output indicating that the Streamlit app is running. Look for a link provided by Colab, or check the 'Ports' tab (usually located in the right-hand panel in Colab, next to 'Files') for a direct link to your running Streamlit application. Click that link to interact with your dashboard.

To stop the Streamlit app, you can interrupt the execution of the Python cell running `subprocess.Popen`.

In [30]:
!pip install streamlit -q
!streamlit run blotter_app.py &>/dev/null &